In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from tqdm.notebook import tqdm_notebook
import gc

from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.hf_api import RepoFolder

In [ ]:
repo_id = "AISC-Linear-Probe-Gen/obfuscated_activations"
max_files = 1e6
num_layers = 28

api = HfApi()

# Auto-discover all folders in the repo
top_level = list(api.list_repo_tree(repo_id=repo_id, repo_type="dataset"))
folders = sorted([item.path for item in top_level if isinstance(item, RepoFolder)])
print(f"Found folders: {folders}")

# Download activations and extract last-position vectors immediately
# all_layers[folder][layer] = numpy array of shape (n_samples, d_model)
all_layers = {}

for folder in folders:
    print(f"\nDownloading from '{folder}'...")
    folder_files = list(api.list_repo_tree(repo_id=repo_id, repo_type="dataset", path_in_repo=folder))
    pt_files = sorted([f.rfilename for f in folder_files if not isinstance(f, RepoFolder) and f.rfilename.endswith(".pt")])

    # Accumulate last-position vectors per layer across files
    per_layer = {layer: [] for layer in range(num_layers)}

    for i, filepath in enumerate(tqdm_notebook(pt_files, desc=folder)):
        if i >= max_files:
            break
        local_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filepath)
        act = torch.load(local_path, weights_only=False)

        for layer in range(num_layers):
            # Extract last position and convert to numpy immediately
            per_layer[layer].append(act[(layer, "layer_out")][:, -1, :].float().numpy())

        del act

    all_layers[folder] = {layer: np.concatenate(vecs) for layer, vecs in per_layer.items()}
    print(f"  Loaded {all_layers[folder][0].shape[0]} samples")

gc.collect()
torch.cuda.empty_cache()
print(f"\nReady — activation types: {list(all_layers.keys())}")

In [ ]:
COLORS = plt.cm.tab10.colors

def plot_pca_for_layer(layer):
    """Plot PCA cumulative variance for all activation types at a given layer."""
    folder_names = list(all_layers.keys())

    plt.figure(figsize=(12, 6))

    # Reference lines
    plt.axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='90% variance', alpha=0.5)
    plt.axhline(y=0.95, color='green', linestyle='--', linewidth=1.5, label='95% variance', alpha=0.5)

    max_components = 0

    for idx, folder in enumerate(folder_names):
        color = COLORS[idx % len(COLORS)]
        data = all_layers[folder][layer]

        pca = PCA()
        pca.fit(data)
        cumvar = np.cumsum(pca.explained_variance_ratio_)
        max_components = max(max_components, len(cumvar))

        n90 = int(np.argmax(cumvar >= 0.90)) + 1
        n95 = int(np.argmax(cumvar >= 0.95)) + 1

        plt.plot(range(1, len(cumvar) + 1), cumvar, linewidth=2, label=folder, color=color)
        plt.axvline(x=n90, color=color, linestyle=':', linewidth=1, alpha=0.4)
        plt.axvline(x=n95, color=color, linestyle=':', linewidth=1, alpha=0.4)

        # Stagger annotation vertically per folder to reduce overlap
        y_offset_90 = 0.87 - 0.03 * idx
        y_offset_95 = 0.96 - 0.03 * idx
        plt.text(n90, y_offset_90, f'{folder}: {n90}', ha='center', va='top', fontsize=8, color=color,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.7))
        plt.text(n95, y_offset_95, f'{folder}: {n95}', ha='center', va='top', fontsize=8, color=color,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.7))

    plt.xlabel('Number of Components', fontsize=12)
    plt.ylabel('Cumulative Variance Explained', fontsize=12)
    plt.title(f'PCA Cumulative Variance — All Activation Types (Layer {layer})', fontsize=14)
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, max_components)
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

    for folder in folder_names:
        data = all_layers[folder][layer]
        pca = PCA()
        pca.fit(data)
        cumvar = np.cumsum(pca.explained_variance_ratio_)
        n90 = int(np.argmax(cumvar >= 0.90)) + 1
        n95 = int(np.argmax(cumvar >= 0.95)) + 1
        print(f"{folder}:  90% → {n90} components,  95% → {n95} components,  total → {len(cumvar)}")

widgets.interact(plot_pca_for_layer, layer=widgets.IntSlider(min=0, max=27, step=1, value=17, description='Layer:'));